# Detecting Generated Texts

### Perturbation Based Methods (DetectGPT)

The most basic curvature based approaches use perturbation to find neighbouring text, used by e.g.

* Mitchell et al. (literature/2301.11305v2.pdf), the OG DetectGPT paper, eq. 1.
* Mireshghallah et al. (literature/2305.09859v4.pdf), the paper about smaller models being better, eq. 1.
* Bao et al. (literature/2310.05130v3.pdf), the FastDetectGPT paper, eq. 3, where it's presented as their DetectGPT baseline.

Given a text $x$, they use a perturbation language model to generate $k$ perturbations of $x$, i.e. similar neighbour texts, $\{\, \tilde{x}_i \,\}_{i=1}^{k}$, and then compare the log-likelihood of the original texts under scoring model $p_{\text{score}}(x) = \prod_t p_{\text{score, token}}(x_t | x_{<t})$ against the average of the perturbed texts. This $d$ can be interpreted as the local curvature or optimality of the log-likelihood. 

$$
d_{\text{pert}} = \log p_{\text{score}}(x) - \frac{1}{k}\sum_{i=1}^{k} \log p_{\text{score}}(\tilde{x}_i)
$$

This is what `PerturbationDetector(metric="sum", normalize_by="none")` implements.

Mitchell et al. also give a std-normalized version, in Algorithm 1. This makes sure that $d(x)$ and $d(y)$ are comparable even if $x$ and $y$ don't have the same sequence length. 

$$
\begin{align*}
\tilde\mu &= \frac{1}{k}\sum_{i=1}^{k} \log p_{\text{score}}(\tilde x_i) \\
\tilde\sigma^2 &= \frac{1}{k-1}\sum_{i=1}^{k} \left(\log p_{\text{score}}(\tilde x_i) - \tilde\mu\right)^2 \\
d_{\text{pert}, \text{norm}} &= \frac{\log p_{\text{score}}(x) - \tilde\mu}{\tilde\sigma}
\end{align*}
$$

This is what `PerturbationDetector(metric="sum", normalize_by="std")` implements.

We introduce a third variant, which doesn't use the log-likelihood over the full text, but the average log-likelihood of the tokens. We do this because the perturbator might produce texts of variable sequence length, which adds noise (longer sequence have smaller likelihood).

$$
\begin{align*}
\tilde\mu &= \frac{1}{k}\sum_{i=1}^{k} \frac{1}{|\tilde x_i|}\log p_{\text{score}}(\tilde x_i) \\
\tilde\sigma^2 &= \frac{1}{k-1}\sum_{i=1}^{k} \left(\frac{1}{|\tilde x_i|}\log p_{\text{score}}(\tilde x_i) - \tilde\mu\right)^2 \\
d_{\text{pert}, \text{avg}} &= \frac{\frac{1}{|x|}\log p_{\text{score}}(x) - \tilde\mu}{\tilde\sigma}
\end{align*}
$$

This is `PerturbationDetector(metric="average", normalize_by="std")`

### Conditional Method (FastDetectGPT)

Bao et al. (literature/2310.05130v3.pdf, eq. 3 and eq. 4), from Fast-DetectGPT, propose an alternative way. Here the neighbouring text are generated from a conditional distribution $\tilde x \sim p_{\text{ref}}(\tilde x \mid x) = \prod_t p_{\text{ref,token}}(\tilde x_t \mid x_{<t})$, and then the conditional log-likelihoods under the scoring model are compared $p_{\text{score}}(\tilde x \mid x) = \prod_t p_{\text{score,token}}(\tilde x_t \mid x_{<t})$

$$
\begin{align*}
\tilde\mu &= \mathbb{E}_{\tilde x \sim p_{\text{ref}}(\cdot \mid x)}\!\left[\log p_{\text{score}}(\tilde x \mid x)\right] \\
\tilde\sigma^2 &= \mathbb{E}_{\tilde x \sim p_{\text{ref}}(\cdot \mid x)}\!\left[(\log p_{\text{score}}(\tilde x \mid x) - \tilde\mu)^2\right] \\
d_{\text{fast}} &= \frac{\log p_{\text{score}}(x \mid x) - \tilde\mu}{\tilde\sigma} 
\end{align*}
$$

This is what `FastDetector(mode="analytic")` implements in closed form, and what `FastDetector(mode="sampling")` estimates using sampling techniques.

## Example

In [1]:
from detector.models import load_model, load_tokenizer, forward_logits
from detector.perturbation import PerturbationDetector, _perturb
from detector.fast import FastDetector, _sample

/Users/pepijnvanderklei/miniforge3/envs/mnlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
scoring_name = "sshleifer/tiny-gpt2"
mask_name = "hf-internal-testing/tiny-random-t5" 
device = "cpu"

scoring_model = load_model(scoring_name, "causal", device=device)
scoring_tokenizer = load_tokenizer(scoring_name)
mask_model = load_model(mask_name, "seq2seq", device=device)
mask_tokenizer = load_tokenizer(mask_name)

Loading weights: 100%|██████████| 29/29 [00:00<00:00, 14091.15it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 113/113 [00:00<00:00, 52902.82it/s]


In [3]:
# Hardcoded examples copied from the MULTITuDE dataset (datasets/MULTITuDE/multitude.csv),
examples = [
    {
        "text": "Australia is cutting its catch of southern bluefin tuna by 30 percent after warnings from scientists that stocks are close to collapse. The reduction follows an international seafood conference in South Korea.",
        "label": 0, "multi_label": "human", "split": "train", "language": "en", "length": 32, "source": "MassiveSumm_voanews",
    },
    {
        "text": "At least 15 members of Iraq's security forces have been killed and dozens more injured in a coordinated attack by Islamic State militants in the country's Anbar province.",
        "label": 1, "multi_label": "vicuna-13b", "split": "test", "language": "en", "length": 28, "source": "MULTITuDE_MassiveSumm_voanews",
    },
    {
        "text": "Een Iraanse politieke activist, Heshmatollah Tabarzadi, heeft een videoboodschap gestuurd vanuit de Rajayishahr-gevangenis. Hij zegt: ‘Onderdrukking en geweld kunnen onze beweging niet tegenhouden.’",
        "label": 0, "multi_label": "human", "split": "train", "language": "nl", "length": 23, "source": "MassiveSumm_globalvoices",
    },
    {
        "text": "In Iran, de mensenrechtenactiviste Najaf Zabani houdt zich niet stil, zelfs in het gevangenschap.",
        "label": 1, "multi_label": "Mistral-7B-Instruct-v0.2", "split": "train", "language": "nl", "length": 14, "source": "MULTITuDE_MassiveSumm_globalvoices",
    },
]

In [4]:
perturbation_detector = PerturbationDetector(
    scoring_model=scoring_model, scoring_tokenizer=scoring_tokenizer,
    mask_model=mask_model, mask_tokenizer=mask_tokenizer, device=device,
    n_perturbations=5, metric="average", normalize_by="std", pct_masked=0.15,
)

fast_analytic = FastDetector(
    reference_model=scoring_model, reference_tokenizer=scoring_tokenizer,
    scoring_model=scoring_model, scoring_tokenizer=scoring_tokenizer,
    device=device, mode="analytic", n_samples=None, sample_top_p=None, sample_top_k=None,
)
fast_sampling = FastDetector(
    reference_model=scoring_model, reference_tokenizer=scoring_tokenizer,
    scoring_model=scoring_model, scoring_tokenizer=scoring_tokenizer,
    device=device, mode="sampling", n_samples=200, sample_top_p=None, sample_top_k=None,
)

In [5]:
for example in examples:
    text = example["text"]
    print(example["language"], "label", example["label"])
    print("  perturbation", perturbation_detector.score(text))
    print("  fast analytic", fast_analytic.score(text))
    print("  fast sampling", fast_sampling.score(text))

en label 0
  perturbation 1.2560273441955163
  fast analytic 0.742131233215332
  fast sampling 0.5225791335105896
en label 1
  perturbation 2.8408767864714144
  fast analytic 2.302432060241699
  fast sampling 2.395638942718506
nl label 0
  perturbation -0.7215580032936931
  fast analytic 0.03777874633669853
  fast sampling 0.1307476907968521
nl label 1
  perturbation 1.3036749257946147
  fast analytic 0.7064567804336548
  fast sampling 0.6814373135566711


### `_perturb`

What do the neighbouring text used by the perturbation methods look like?

In [6]:
demo_text = examples[0]["text"]

print("original: ", demo_text)
print("perturbed:", _perturb(demo_text, 1, mask_model, mask_tokenizer, device, 2, 0.15, 1.0, None)[0])

original:  Australia is cutting its catch of southern bluefin tuna by 30 percent after warnings from scientists that stocks are close to collapse. The reduction follows an international seafood conference in South Korea.
perturbed: Australia is 誡&ेςぎΩ珙ة catch of southern bluefin tuna by 30 percent after warnings from scientists îĐܝႣバ礮趙ɪνჶ主古연パゲ×χţ藥園拉म are close to collapse. The reduction follows an international seafood conference in South Korea.


### `_sample`

What do the neighbouring texts generated by the conditional approach from FastDetectGPT look like?

In [7]:
demo_logits_ref, demo_labels = forward_logits(demo_text, scoring_model, scoring_tokenizer, device)
resampled = _sample(demo_logits_ref, 1, None, None)

print("original:", scoring_tokenizer.decode(demo_labels[0]))
print(f"sample:", scoring_tokenizer.decode(resampled[0, :, 0]))

original:  is cutting its catch of southern bluefin tuna by 30 percent after warnings from scientists that stocks are close to collapse. The reduction follows an international seafood conference in South Korea.
sample:  chore lizard 1863 simulations Basil IOCtrust Brendachangesyre136Address Surveydeals .............. paydaySetuprecstracterocontained evident Census aspire,- Problems referred Intellectual missionary Jay infuriicaダ SPEC
